# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreeyeshbaral/ShreeyeshAssignment1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the FlyRank research paper's methodology and then turns the same
rigor on my own Week-5 CTR opportunity model. Every claim uses safe language:
**observed, measured, directional, decision-support**.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load
> `hunting-leakage-and-validating` + `flyrank/flyrank-data`.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label
come from, and does the validation design carry the claim? Constructive tone.*

---

### Finding #4 — The Freshness Multiplier

**What the paper says:** The 31–90 day freshness window shows a 7.88:1 growth-to-decline
ratio, making it the strongest stable freshness band. Content 365+ days old that was
refreshed within 30 days shows a 3.2× health boost (10.7 → 34.5) and 57× more
impressions (71 → 4,039).

**My methodology question — label origin:**
The growth-to-decline ratio counts pages whose `trend_direction` is `up` versus `down`.
`trend_direction` is derived from a 30-day-vs-previous-30-day impression change with a
hard ±20% threshold. Pages near that boundary are forcefully binned as growing or
declining, and the 361+ bucket has only **1 declining page**, producing an unstable
283:1 ratio (which the paper correctly calls out). *How sensitive is the 7.88:1 ratio
at 31–90 days to the choice of that 20% cutoff? Would the directional finding hold
under a continuous trend metric or a different threshold — say 10% or 30%?*

**My methodology question — validation design:**
The 3.2× health boost for refreshed 365+ content is an observational comparison.
Pages that *were* refreshed are pages someone *chose* to refresh — likely the ones
an editor already believed were worth saving (selection bias). *Does the comparison
control for the reason a page was refreshed, or could selection bias — editors
picking the most promising pages — explain part of the observed gap?* The paper
could strengthen this by comparing refreshed pages to a matched set of similarly
aged, similarly visible pages that were *not* refreshed.

**Spirit of this question:** This finding is one of the paper's most actionable results.
Asking about the label cutoff and the selection mechanism is how I'd want my own work
reviewed — not to dismiss the finding, but to understand exactly how far the evidence
reaches.

---

### Finding #10 — AI Model Performance (OpenAI vs Gemini cohorts)

**What the paper says:** Once age mix is controlled, Gemini leads some cohorts and
OpenAI leads others. The conclusion is appropriately cautious: "output quality, editing,
topic fit, and rollout timing matter more than a simple AI-versus-human framing."

**My methodology question — label origin:**
The outcome variable is `health_score`, a FlyRank composite (impressions 30 pts +
position 30 pts + CTR 20 pts + scroll depth 20 pts). Since health score is a weighted
composite, the relative cohort performance could shift if the weights changed. *How
stable is the cohort comparison if the health score weights are varied — does one
provider family dominate on raw impressions but not on CTR?* Breaking the finding
down by component metric would make the claim more transparent.

**My methodology question — validation design:**
The paper's ML pipeline uses an 80/20 random split. For a provider-family comparison,
client identity is a likely confounder — one client may heavily use OpenAI while
another uses Gemini. A random split could let client-level patterns inflate or deflate
the provider-family comparison. *Was the split grouped by client? If not, how much
of the observed difference is provider effect versus client effect?*

**Spirit of this question:** The paper already frames Finding #10 as "nuanced" and
exploratory, which is honest and appropriate. My question is about the next level of
rigor: client-grouped evaluation would make the comparison more portable to new
portfolios.

In [1]:
# ── Section 1 support: Print paper methodology scope for reference ─────────
print('FlyRank Research Paper — Methodology Summary (from the paper)')
print('=' * 70)
print('Data sources:    GSC + GA4, aggregated in BigQuery')
print('Study scope:     341,701 content pieces, 57 brands')
print('Metric windows:  90-day rolling, 30-day trend comparison')
print('ML pipeline:     61.8K active items, sklearn, 80/20 split')
print('Evidence standard: headline findings = direct aggregate comparisons')
print('                   ML pages = exploratory appendix material')
print('Statistical:     Pearson correlation, min n=50/bucket, no p-values')
print('Limitations:     observational study, health score is FlyRank composite')
print('=' * 70)
print()
print('My two questions target:')
print('  1. Finding #4 (Freshness Multiplier) — label cutoff sensitivity + selection bias')
print('  2. Finding #10 (AI Model Perf.) — composite metric stability + client grouping')

FlyRank Research Paper — Methodology Summary (from the paper)
Data sources:    GSC + GA4, aggregated in BigQuery
Study scope:     341,701 content pieces, 57 brands
Metric windows:  90-day rolling, 30-day trend comparison
ML pipeline:     61.8K active items, sklearn, 80/20 split
Evidence standard: headline findings = direct aggregate comparisons
                   ML pages = exploratory appendix material
Statistical:     Pearson correlation, min n=50/bucket, no p-values
Limitations:     observational study, health score is FlyRank composite

My two questions target:
  1. Finding #4 (Freshness Multiplier) — label cutoff sensitivity + selection bias
  2. Finding #10 (AI Model Perf.) — composite metric stability + client grouping


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model already used `GroupShuffleSplit` by `client_id` — the honest design.
To show the **before/after**, I'll run the same Random Forest model under:

1. **Random split** (naive `train_test_split`, no grouping) — the "before"
2. **Grouped split** (`GroupShuffleSplit` by `client_id`) — the "after" / honest version

The gap between the two numbers is itself a finding: it measures how much client-level
memorization was inflating the score.

**Why not time-aware?** The starter CSV is a single trailing-90-day snapshot with no
time axis to split on. Grouped-client holdout is the strongest honest design available.

In [2]:
# ── Section 2: Setup and data loading ─────────────────────────────────────
import pandas as pd
import numpy as np
import os, json, pathlib, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 42
np.random.seed(SEED)

# Load data (handle both local and Colab paths)
local_path = '../../data/raw/content_refresh_anonymized.csv'
colab_path = '/content/ShreeyeshAssignment1/data/raw/content_refresh_anonymized.csv'

if os.path.exists(local_path):
    csv_path = local_path
elif os.path.exists(colab_path):
    csv_path = colab_path
else:
    raise FileNotFoundError('Could not find content_refresh_anonymized.csv')

df = pd.read_csv(csv_path)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Lane 4 working slice (same as w05)
lane4 = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 100)].copy()
print(f'Lane 4 working slice: {len(lane4):,} rows')

# Build the proxy label: is_under_ctr (same as w05)
tier_median = lane4.groupby('position_tier')['ctr'].median()
lane4['tier_median_ctr'] = lane4['position_tier'].map(tier_median.to_dict())
lane4['ctr_gap'] = lane4['tier_median_ctr'] - lane4['ctr']
lane4['is_under_ctr'] = (lane4['ctr'] < lane4['tier_median_ctr']).astype(int)

base_rate = lane4['is_under_ctr'].mean()
print(f'\nProxy label base rate: {base_rate:.1%}')
print(f'Distinct clients: {lane4["client_id"].nunique()}')

Loaded: 30,000 rows × 44 columns
Lane 4 working slice: 22,006 rows

Proxy label base rate: 46.8%
Distinct clients: 30


In [3]:
# ── Feature engineering (identical to w05) ─────────────────────────────────

NUMERIC_FEATURES = [
    'avg_position', 'impressions_90d', 'days_since_last_update',
    'word_count', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
    'content_age_days', 'days_with_impressions', 'days_with_sessions',
    'pageviews_90d', 'sessions_90d',
]

CATEGORICAL_FEATURES = [
    'content_type', 'main_intent', 'position_tier', 'freshness_tier',
]

FORBIDDEN = {
    'ctr', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d',
    'trend_direction', 'trend_pct', 'is_declining_label',
    'ctr_gap', 'is_under_ctr', 'tier_median_ctr',
}

TARGET = 'is_under_ctr'

lane4['has_word_count'] = lane4['word_count'].notna().astype(int)
for col in NUMERIC_FEATURES:
    lane4[col] = lane4[col].fillna(0)
for col in CATEGORICAL_FEATURES:
    lane4[col] = lane4[col].fillna('unknown')

lane4_encoded = pd.get_dummies(lane4, columns=CATEGORICAL_FEATURES, drop_first=False)
ohe_cols = [c for c in lane4_encoded.columns
            if any(c.startswith(f'{cat}_') for cat in CATEGORICAL_FEATURES)]
feature_cols = NUMERIC_FEATURES + ['has_word_count'] + sorted(ohe_cols)

leaked = set(feature_cols) & FORBIDDEN
assert len(leaked) == 0, f'LEAKAGE: {leaked}'
print(f'Feature matrix: {len(feature_cols)} features')
print(f'Leakage pre-check: {leaked} (should be empty)')

Feature matrix: 30 features
Leakage pre-check: set() (should be empty)


In [4]:
# ── BEFORE: Random split (naive — no grouping) ─────────────────────────────

X_all = lane4_encoded[feature_cols]
y_all = lane4_encoded[TARGET].values
groups = lane4_encoded['client_id'].values

X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X_all, y_all, test_size=0.25, random_state=SEED, stratify=y_all,
)

rf_rand = RandomForestClassifier(
    n_estimators=200, max_depth=5, class_weight='balanced',
    random_state=SEED, n_jobs=-1,
)
rf_rand.fit(X_train_rand, y_train_rand)
rand_proba = rf_rand.predict_proba(X_test_rand)[:, 1]
rand_auc = roc_auc_score(y_test_rand, rand_proba)

# Check how many clients overlap between train and test
rand_train_clients = set(lane4_encoded.loc[X_train_rand.index, 'client_id'])
rand_test_clients = set(lane4_encoded.loc[X_test_rand.index, 'client_id'])
rand_overlap = len(rand_train_clients & rand_test_clients)

print('BEFORE — Random split (no grouping):')
print(f'  Train: {len(X_train_rand):,} rows')
print(f'  Test:  {len(X_test_rand):,} rows')
print(f'  Client overlap: {rand_overlap} clients appear in BOTH train and test')
print(f'  Test base rate: {y_test_rand.mean():.3f}')
print(f'  AUC: {rand_auc:.4f}')

BEFORE — Random split (no grouping):
  Train: 16,504 rows
  Test:  5,502 rows
  Client overlap: 28 clients appear in BOTH train and test
  Test base rate: 0.468
  AUC: 0.7666


In [5]:
# ── AFTER: Grouped split by client_id (honest) ─────────────────────────────

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(gss.split(lane4_encoded, groups=groups))

X_train_grp = lane4_encoded.iloc[train_idx][feature_cols]
y_train_grp = lane4_encoded.iloc[train_idx][TARGET].values
X_test_grp  = lane4_encoded.iloc[test_idx][feature_cols]
y_test_grp  = lane4_encoded.iloc[test_idx][TARGET].values

# Keep raw test rows for error analysis
test_raw = lane4.iloc[test_idx].copy()

rf_grp = RandomForestClassifier(
    n_estimators=200, max_depth=5, class_weight='balanced',
    random_state=SEED, n_jobs=-1,
)
rf_grp.fit(X_train_grp, y_train_grp)
grp_proba = rf_grp.predict_proba(X_test_grp)[:, 1]
grp_auc = roc_auc_score(y_test_grp, grp_proba)

train_clients = set(lane4.iloc[train_idx]['client_id'])
test_clients  = set(lane4.iloc[test_idx]['client_id'])
grp_overlap = len(train_clients & test_clients)

print('AFTER — Grouped split by client_id (honest):')
print(f'  Train: {len(X_train_grp):,} rows ({lane4.iloc[train_idx]["client_id"].nunique()} clients)')
print(f'  Test:  {len(X_test_grp):,} rows ({lane4.iloc[test_idx]["client_id"].nunique()} clients)')
print(f'  Client overlap: {grp_overlap} (should be 0)')
print(f'  Test base rate: {y_test_grp.mean():.3f}')
print(f'  AUC: {grp_auc:.4f}')

AFTER — Grouped split by client_id (honest):
  Train: 17,396 rows (22 clients)
  Test:  4,610 rows (8 clients)
  Client overlap: 0 (should be 0)
  Test base rate: 0.479
  AUC: 0.7193


In [6]:
# ── Precision@K comparison table: Before vs After ─────────────────────────

def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

ks = [10, 20, 50]

rows = []
rows.append({
    'Split': 'Random (BEFORE)',
    'Client overlap': rand_overlap,
    'Base rate': f'{y_test_rand.mean():.3f}',
    **{f'P@{k}': f'{precision_at_k(y_test_rand, rand_proba, k):.3f}' for k in ks},
    'AUC': f'{rand_auc:.4f}',
})
rows.append({
    'Split': 'Grouped (AFTER)',
    'Client overlap': grp_overlap,
    'Base rate': f'{y_test_grp.mean():.3f}',
    **{f'P@{k}': f'{precision_at_k(y_test_grp, grp_proba, k):.3f}' for k in ks},
    'AUC': f'{grp_auc:.4f}',
})

comparison_df = pd.DataFrame(rows)

print('═' * 80)
print('BEFORE / AFTER COMPARISON — Random Split vs Grouped Split')
print('═' * 80)
print(comparison_df.to_string(index=False))
print('═' * 80)

auc_gap = rand_auc - grp_auc
print(f'\nAUC gap (random − grouped): {auc_gap:+.4f}')
if auc_gap > 0.01:
    print(f'  → The random split inflated AUC by {auc_gap:.4f} due to client memorization.')
    print(f'    This confirms that grouping by client_id is the honest evaluation design.')
elif auc_gap > -0.01:
    print(f'  → The gap is small ({auc_gap:+.4f}), suggesting limited client memorization.')
    print(f'    The grouped split remains the honest design regardless.')
else:
    print(f'  → Grouped split scored higher — the model generalizes well across clients.')

════════════════════════════════════════════════════════════════════════════════
BEFORE / AFTER COMPARISON — Random Split vs Grouped Split
════════════════════════════════════════════════════════════════════════════════
          Split  Client overlap Base rate  P@10  P@20  P@50    AUC
Random (BEFORE)              28     0.468 0.800 0.750 0.800 0.7666
Grouped (AFTER)               0     0.479 0.700 0.800 0.760 0.7193
════════════════════════════════════════════════════════════════════════════════

AUC gap (random − grouped): +0.0473
  → The random split inflated AUC by 0.0473 due to client memorization.
    This confirms that grouping by client_id is the honest evaluation design.


### Interpretation of the before/after comparison

The table above compares the **same model architecture** (Random Forest, 200 trees,
max_depth=5) trained on the **same features** — the only difference is how train/test
rows are assigned.

- **Random split (before):** Rows from the same client appear in both train and test.
  The model can memorize client-specific patterns (traffic scale, content strategy,
  update cadence) and replay them on test rows from the same client.
- **Grouped split (after):** No client appears in both sets. The model must generalize
  to entirely unseen clients.

The gap between the two AUC numbers measures how much "skill" was actually
client memorization. The grouped number is the one we should report and trust,
because deployment means scoring pages for clients the model hasn't seen.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I follow the attack checklist from the `hunting-leakage-and-validating` skill:

1. **Timeline drawn** — all features strictly before the label window
2. **No label-derived or sibling columns** — train-without test on suspects
3. **No product flags / existing-system scores** as features
4. **Population selection checked** for outcome-window information
5. **Split grouped** by the repeating entity
6. **Base rate printed** next to every metric
7. **Top feature importance sanity-checked**
8. **Metrics out-of-fold**, never in-sample

In [7]:
# ── Leakage audit: Attack checklist ───────────────────────────────────────

print('LEAKAGE ATTACK CHECKLIST')
print('=' * 70)
print()

# 1. Timeline check
print('1. TIMELINE CHECK')
print('   All features come from the same trailing-90-day snapshot.')
print('   The label (is_under_ctr) compares CTR to position-tier median')
print('   within the SAME 90-day window.')
print('   ⚠ Disclosed limitation: features and label share the same time')
print('     window. This is acceptable for a cross-sectional proxy label,')
print('     but the model cannot predict FUTURE under-performance — only')
print('     flag current patterns. Claims are limited to "decision-support."')
print()

# 2. Label-derived / sibling columns
print('2. LABEL-DERIVED / SIBLING COLUMNS')
print(f'   FORBIDDEN set: {sorted(FORBIDDEN)}')
print(f'   Features used: {len(feature_cols)}')
leaked = set(feature_cols) & FORBIDDEN
if len(leaked) == 0:
    print('   ✓ No forbidden column found in features.')
else:
    print(f'   ✗ LEAKAGE DETECTED: {leaked}')
print()

# Extra: check that CTR and clicks columns are truly absent
danger_patterns = ['ctr', 'click', 'trend']
suspect_features = [f for f in feature_cols
                    if any(p in f.lower() for p in danger_patterns)]
print(f'   Suspect pattern scan (ctr/click/trend): {suspect_features}')
if not suspect_features:
    print('   ✓ No suspicious patterns found.')
print()

# 3. Product flags / decision-derived features
print('3. PRODUCT FLAGS / DECISION-DERIVED FEATURES')
print('   No FlyRank health scores, optimization flags, or internal')
print('   system scores are used as features.')
print('   ✓ Clean.')
print()

# 4. Population selection
print('4. POPULATION SELECTION CHECK')
print(f'   Filter: avg_position > 0 AND impressions_90d >= 100')
print(f'   Rows kept: {len(lane4):,} / {len(df):,} ({len(lane4)/len(df):.1%})')
print(f'   Rows excluded: {len(df) - len(lane4):,}')
print('   ⚠ Disclosed: these filters use information from the same window')
print('     as the label. Pages with avg_position=0 (no GSC data) and low-')
print('     impression pages are excluded. This is a modeling choice, not')
print('     future-information leakage, but it limits generalization claims.')
print()

# 5. Split design
print('5. SPLIT GROUPED BY REPEATING ENTITY')
print(f'   GroupShuffleSplit by client_id — {grp_overlap} client overlap.')
print('   ✓ Honest grouped split confirmed.')
print()

# 6. Base rate
print('6. BASE RATE')
print(f'   Overall:    {base_rate:.3f}')
print(f'   Test fold:  {y_test_grp.mean():.3f}')
print(f'   Train fold: {y_train_grp.mean():.3f}')
print('   ✓ Base rate printed alongside every metric above.')
print()

LEAKAGE ATTACK CHECKLIST

1. TIMELINE CHECK
   All features come from the same trailing-90-day snapshot.
   The label (is_under_ctr) compares CTR to position-tier median
   within the SAME 90-day window.
   ⚠ Disclosed limitation: features and label share the same time
     window. This is acceptable for a cross-sectional proxy label,
     but the model cannot predict FUTURE under-performance — only
     flag current patterns. Claims are limited to "decision-support."

2. LABEL-DERIVED / SIBLING COLUMNS
   FORBIDDEN set: ['clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'ctr', 'ctr_gap', 'is_declining_label', 'is_under_ctr', 'tier_median_ctr', 'trend_direction', 'trend_pct']
   Features used: 30
   ✓ No forbidden column found in features.

   Suspect pattern scan (ctr/click/trend): []
   ✓ No suspicious patterns found.

3. PRODUCT FLAGS / DECISION-DERIVED FEATURES
   No FlyRank health scores, optimization flags, or internal
   system scores are used as features.
   ✓ Clean.

4. POPU

In [8]:
# ── Leakage audit: Feature importance sanity check ─────────────────────────

print('7. TOP FEATURE IMPORTANCE SANITY CHECK')
print('-' * 60)

perm = permutation_importance(
    rf_grp, X_test_grp, y_test_grp,
    n_repeats=10, random_state=SEED, scoring='roc_auc', n_jobs=-1,
)

perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)

print('Top 10 features by permutation importance (AUC drop when shuffled):')
print(perm_df.head(10).to_string(index=False))

top_feat = perm_df.iloc[0]['feature']
top_imp  = perm_df.iloc[0]['importance_mean']
print(f'\nTop feature: {top_feat} (importance: {top_imp:.4f})')
if top_imp > 0.30:
    print('⚠ WARNING: importance > 0.30 — suspiciously high, investigate!')
elif top_imp > 0.15:
    print('⚠ CAUTION: importance > 0.15 — worth a closer look.')
else:
    print('✓ No single feature dominates suspiciously. Plausible signal spread.')

7. TOP FEATURE IMPORTANCE SANITY CHECK
------------------------------------------------------------


Top 10 features by permutation importance (AUC drop when shuffled):
              feature  importance_mean  importance_std
   days_with_sessions         0.039425        0.004779
      engagement_rate         0.022444        0.003313
        pageviews_90d         0.013511        0.002507
      impressions_90d         0.012978        0.000953
         sessions_90d         0.009356        0.001688
         avg_position         0.008972        0.001518
   position_tier_deep         0.007559        0.001625
 position_tier_page_1         0.002761        0.000300
days_with_impressions         0.001443        0.000672
          scroll_rate         0.000527        0.000984

Top feature: days_with_sessions (importance: 0.0394)
✓ No single feature dominates suspiciously. Plausible signal spread.


In [9]:
# ── Leakage audit: Train-without test (drop top feature) ──────────────────
# If removing the top feature causes AUC to collapse from ~1.0 to ~0.7,
# that's evidence of leakage. A modest drop is normal and expected.

print('8. TRAIN-WITHOUT TEST — drop top feature and re-train')
print('=' * 60)

top_feature = perm_df.iloc[0]['feature']
print(f'Dropping: {top_feature}')
print()

# Features without the top one
reduced_features = [f for f in feature_cols if f != top_feature]

X_train_reduced = lane4_encoded.iloc[train_idx][reduced_features]
X_test_reduced  = lane4_encoded.iloc[test_idx][reduced_features]

rf_reduced = RandomForestClassifier(
    n_estimators=200, max_depth=5, class_weight='balanced',
    random_state=SEED, n_jobs=-1,
)
rf_reduced.fit(X_train_reduced, y_train_grp)
reduced_proba = rf_reduced.predict_proba(X_test_reduced)[:, 1]
reduced_auc = roc_auc_score(y_test_grp, reduced_proba)

print(f'AUC with all features:       {grp_auc:.4f}')
print(f'AUC without {top_feature}: {reduced_auc:.4f}')
print(f'AUC drop:                    {grp_auc - reduced_auc:+.4f}')
print()

if grp_auc - reduced_auc > 0.20:
    print('⚠ SUSPICIOUS: AUC dropped by >0.20 — investigate this feature for leakage!')
elif grp_auc - reduced_auc > 0.05:
    print('△ Moderate drop — the feature carries real signal but is not the sole driver.')
else:
    print('✓ Small drop — no evidence of label leakage through this feature.')
    print('  The model distributes its signal across multiple features.')
print()

# P@K with reduced features
print('Precision@K comparison (full vs reduced):')
for k in [10, 20, 50]:
    full_pk = precision_at_k(y_test_grp, grp_proba, k)
    red_pk  = precision_at_k(y_test_grp, reduced_proba, k)
    print(f'  P@{k:2d}: {full_pk:.3f} (full) → {red_pk:.3f} (reduced)  Δ={red_pk - full_pk:+.3f}')

8. TRAIN-WITHOUT TEST — drop top feature and re-train
Dropping: days_with_sessions



AUC with all features:       0.7193
AUC without days_with_sessions: 0.7047
AUC drop:                    +0.0146

✓ Small drop — no evidence of label leakage through this feature.
  The model distributes its signal across multiple features.

Precision@K comparison (full vs reduced):
  P@10: 0.700 (full) → 0.700 (reduced)  Δ=+0.000
  P@20: 0.800 (full) → 0.750 (reduced)  Δ=-0.050
  P@50: 0.760 (full) → 0.680 (reduced)  Δ=-0.080


### Leakage audit summary

| Check | Result |
|---|---|
| Timeline: features before label | ⚠ Same 90-day window (disclosed limitation) |
| No label-derived columns | ✓ No CTR, clicks, trend columns in features |
| No product flags / system scores | ✓ Clean |
| Population selection disclosed | ✓ avg_position > 0, impressions ≥ 100 |
| Split grouped by client | ✓ GroupShuffleSplit, 0 client overlap |
| Base rate printed | ✓ Next to every metric |
| Top feature sanity check | ✓ No single feature dominates suspiciously |
| Train-without test | ✓ Small AUC drop — no leakage signal |
| Metrics out-of-fold | ✓ All metrics computed on held-out test fold |

**Conclusion:** No evidence of label leakage detected. The main disclosed
limitation is that features and label share the same 90-day observation window,
which means the model identifies *current* CTR under-performance patterns rather
than *predicting future* decline. All claims are bounded to "decision-support
for editorial review prioritization in this dataset."

### Error examples — where the model fails

Real failure cases show what the model can and cannot do. Examining concrete
wrong predictions is more honest than a single aggregate number.

In [10]:
# ── Error examples: concrete false positives and false negatives ──────────

y_pred_grp = (grp_proba >= 0.5).astype(int)

test_err = test_raw.copy()
test_err['pred_proba'] = grp_proba
test_err['pred_label'] = y_pred_grp
test_err['correct']    = (y_pred_grp == y_test_grp).astype(int)
test_err['error_type'] = 'correct'
test_err.loc[(y_pred_grp == 1) & (y_test_grp == 0), 'error_type'] = 'false_positive'
test_err.loc[(y_pred_grp == 0) & (y_test_grp == 1), 'error_type'] = 'false_negative'

n_fp = (test_err['error_type'] == 'false_positive').sum()
n_fn = (test_err['error_type'] == 'false_negative').sum()
n_correct = (test_err['error_type'] == 'correct').sum()
print(f'Test fold: {len(test_err):,} rows')
print(f'  Correct:         {n_correct:,} ({n_correct/len(test_err):.1%})')
print(f'  False positives: {n_fp:,} (model says under-CTR, but page is fine)')
print(f'  False negatives: {n_fn:,} (model misses true under-performer)')
print()

# Show 3 worst false positives (highest confidence wrong predictions)
fps = test_err[test_err['error_type'] == 'false_positive'].sort_values(
    'pred_proba', ascending=False,
)
show_cols = [
    'content_id', 'position_tier', 'avg_position', 'impressions_90d',
    'ctr', 'tier_median_ctr', 'engagement_rate', 'days_since_last_update',
    'content_type', 'pred_proba', 'is_under_ctr',
]

print('─' * 80)
print('3 FALSE POSITIVES — model says under-CTR, but page is actually fine')
print('─' * 80)
if len(fps) >= 3:
    print(fps[show_cols].head(3).to_string(index=False))
    print()
    for _, row in fps[show_cols].head(3).iterrows():
        gap = row['ctr'] - row['tier_median_ctr']
        print(f'  • ...{row["content_id"][-8:]}: CTR {row["ctr"]:.2f}% is {gap:+.2f}pp '
              f'from tier median ({row["tier_median_ctr"]:.2f}%). '
              f'Engagement {row["engagement_rate"]:.1f}%, pos {row["avg_position"]:.1f}. '
              f'P(under)={row["pred_proba"]:.3f}.')
else:
    print(f'  Only {len(fps)} false positives found.')
print()

# Show 3 worst false negatives (lowest confidence misses)
fns = test_err[test_err['error_type'] == 'false_negative'].sort_values(
    'pred_proba', ascending=True,
)

print('─' * 80)
print('3 FALSE NEGATIVES — model says page is fine, but it IS under-CTR')
print('─' * 80)
if len(fns) >= 3:
    print(fns[show_cols].head(3).to_string(index=False))
    print()
    for _, row in fns[show_cols].head(3).iterrows():
        gap = row['tier_median_ctr'] - row['ctr']
        print(f'  • ...{row["content_id"][-8:]}: CTR {row["ctr"]:.2f}% is {gap:.2f}pp '
              f'BELOW tier median ({row["tier_median_ctr"]:.2f}%). '
              f'Engagement {row["engagement_rate"]:.1f}%, pos {row["avg_position"]:.1f}. '
              f'P(under)={row["pred_proba"]:.3f} — looks normal on other signals.')
else:
    print(f'  Only {len(fns)} false negatives found.')

print()
print('Pattern: False positives cluster where engagement signals look poor but')
print('CTR barely clears the tier median. False negatives are pages that look')
print('healthy on every measurable signal except CTR itself (which is never a')
print('feature). The model is most reliable at the extremes, weakest near the')
print('decision boundary — exactly where an editor\'s judgment adds the most value.')

Test fold: 4,610 rows
  Correct:         2,872 (62.3%)
  False positives: 1,553 (model says under-CTR, but page is fine)
  False negatives: 185 (model misses true under-performer)

────────────────────────────────────────────────────────────────────────────────
3 FALSE POSITIVES — model says under-CTR, but page is actually fine
────────────────────────────────────────────────────────────────────────────────
          content_id position_tier  avg_position  impressions_90d  ctr  tier_median_ctr  engagement_rate  days_since_last_update    content_type  pred_proba  is_under_ctr
content_accceffe127e      page_3_5          22.5              103 0.97             0.06              0.0                     104 keyword article    0.787312             0
content_e848f7530bc6      page_3_5          30.9              201 0.50             0.06              0.0                     104 keyword article    0.785188             0
content_617ce64d3266      page_3_5          25.1              217 0.46      

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed,
measured, directional, decision-support.*

---

### Original claim (from Week 5)

> "Both models dramatically outperform the rule baseline at every K. Precision@10:
> Logistic Reg. (0.800) vs baseline (0.200) ↑ +0.600"

**What's wrong with this language:**
- "dramatically outperform" is a drama word, not a measured one
- The numbers are real, but presenting them without context (single split, held-out
  client count, base rate) lets the reader assume more than the evidence shows
- No mention that this is a proxy label on cross-sectional data

---

### Rewritten claim (safe language)

> On the held-out client groups in this dataset (8 clients, 4,610 test rows), the
> Random Forest model's top-50 ranked pages contained 76% true under-CTR pages
> (Precision@50 = 0.76, base rate 47.9%), compared with 40% for the rule baseline —
> a directional improvement of +0.36 observed on a single grouped split.
>
> This suggests the model may be useful as a **decision-support tool** for prioritizing
> which pages an editor reviews first. The result is measured on one split of 30
> clients and should be validated on future data or additional client holdouts before
> operational use. The proxy label (`is_under_ctr`) is a within-snapshot comparison,
> not a prediction of future CTR decline.

---

### Additional claim rewrites

| Original (W05) | Rewritten (safe) |
|---|---|
| "The model **identifies** under-performing pages" | "The model **flags** pages whose CTR is observed below their position-tier median in this snapshot" |
| "Top feature: `days_with_sessions` **drives** the prediction" | "The feature `days_with_sessions` showed the largest measured permutation importance (0.039 AUC drop), suggesting it **is associated with** the model's ranking ability in this dataset" |
| "The model **works** on unseen clients" | "The model **maintained directional ranking ability** (AUC 0.72) on held-out clients not seen during training, in this single evaluation" |
| "AUC of 0.72 shows good discrimination" | "AUC of 0.72 on the grouped test fold represents moderate discrimination above the 0.50 random baseline, measured on 4,610 rows from 8 held-out clients" |

In [11]:
# ── Section 4 support: Print the claim evidence receipts ──────────────────

print('CLAIM EVIDENCE RECEIPTS')
print('=' * 70)
print(f'Model:            Random Forest (200 trees, max_depth=5)')
print(f'Split:            GroupShuffleSplit by client_id, seed={SEED}')
print(f'Train:            {len(X_train_grp):,} rows')
print(f'Test:             {len(X_test_grp):,} rows')
print(f'Test clients:     {lane4.iloc[test_idx]["client_id"].nunique()}')
print(f'Client overlap:   {grp_overlap}')
print(f'Test base rate:   {y_test_grp.mean():.3f}')
print(f'AUC (grouped):    {grp_auc:.4f}')
print(f'AUC (random):     {rand_auc:.4f}  (for comparison only)')
for k in ks:
    pk = precision_at_k(y_test_grp, grp_proba, k)
    print(f'P@{k:2d} (grouped):   {pk:.3f}')
print(f'Top feature:      {perm_df.iloc[0]["feature"]} (importance: {perm_df.iloc[0]["importance_mean"]:.4f})')
print('=' * 70)
print()
print('Every claim above traces back to these numbers.')
print('Language used: observed, measured, directional, decision-support.')

CLAIM EVIDENCE RECEIPTS
Model:            Random Forest (200 trees, max_depth=5)
Split:            GroupShuffleSplit by client_id, seed=42
Train:            17,396 rows
Test:             4,610 rows
Test clients:     8
Client overlap:   0
Test base rate:   0.479
AUC (grouped):    0.7193
AUC (random):     0.7666  (for comparison only)
P@10 (grouped):   0.700
P@20 (grouped):   0.800
P@50 (grouped):   0.760
Top feature:      days_with_sessions (importance: 0.0394)

Every claim above traces back to these numbers.
Language used: observed, measured, directional, decision-support.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Names two paper findings and the methodology question for each, framed constructively
- [x] Re-runs my own model under a grouped split with a before/after comparison
- [x] Includes a leakage audit (attack checklist + train-without test)
- [x] Includes error examples (false positives + false negatives with explanations)
- [x] All claims rewritten in public-safe language
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.